## Notebook 4 - Gold Layer Notebook

### Load Silver Delta Tables

In [ ]:
from pyspark.sql import functions as F
from pyspark.sql.window import Window

electricity = spark.table("carbonops.silver.electricity_enriched")
fuel = spark.table("carbonops.silver.fuel_enriched")
production = spark.table("carbonops.silver.production_clean")
emission_factors = spark.table("carbonops.silver.emission_factors_clean")
carbon_targets = spark.table("carbonops.silver.carbon_targets_clean")

print("CarbonOps Silver tables loaded successfully!")

### Calculate Scope 2 Emissions (Electricity)

Scope 2 emissions represent indirect emissions from purchased electricity.

For our synthetic dataset:
$$
\text{Scope 2 (tCO}_2\text{e)} = \frac{\text{Electricity (kWh)} \times \text{Emission Factor}}{1000}
$$

We'll join electricity consumption with the emission factors based on year, geography, source type, and units.

In [ ]:
electricity_factors = (
    emission_factors
    .filter(F.col("source_type") == "electricity_grid")
    .select(
        F.col("year").alias("factor_year"),
        F.col("geography").alias("factor_geography"),
        F.lower(F.col("activity_unit")).alias("factor_unit"),
        F.col("kg_co2e_per_unit")
    )
)

scope2 = (
    electricity
    .withColumn("year", F.year("period_end"))
    .join(
        electricity_factors,
        (F.col("year") == F.col("factor_year")) &
        (F.col("country") == F.col("factor_geography")) &
        (F.lit("kwh") == F.col("factor_unit")),
        "inner"
    )
    .withColumn(
        "scope2_tco2e",
        F.col("electricity_kwh") *
        F.col("kg_co2e_per_unit") / 1000
    )
    .groupBy("facility_id", "period_end")
    .agg(
        F.sum("electricity_kwh").alias("total_electricity_kwh"),
        F.sum("scope2_tco2e").alias("scope2_tco2e")
    )
)

display(scope2)

### Calculate Scope 1 Emissions (Fuel)

Scope 1 emissions represent direct emissions from sources owned or controlled by the company.

Our dataset includes diesel, natural gas, and LPG consumption.
$$
\text{Scope 1 (tCO}_2\text{e)} = \frac{\sum(\text{Fuel Quantity} \times \text{Emission Factor})}{1000}
$$

In [ ]:
fuel_factors = (
    emission_factors
    .filter(F.col("source_type").isin(
        "diesel", "natural_gas", "lpg"
    ))
    .select(
        F.col("source_type").alias("factor_fuel_type"),
        F.col("year").alias("factor_year"),
        F.col("geography").alias("factor_geography"),
        F.lower(F.col("activity_unit")).alias("factor_unit"),
        F.col("kg_co2e_per_unit")
    )
)

scope1 = (
    fuel
    .withColumn("year", F.year("period_end"))
    .join(
        fuel_factors,
        (F.col("fuel_type") == F.col("factor_fuel_type")) &
        (F.col("year") == F.col("factor_year")) &
        (F.col("country") == F.col("factor_geography")) &
        (F.col("unit") == F.col("factor_unit")),
        "inner"
    )
    .withColumn(
        "scope1_tco2e",
        F.col("quantity") *
        F.col("kg_co2e_per_unit") / 1000
    )
    .groupBy("facility_id", "period_end")
    .agg(
        F.sum("scope1_tco2e").alias("scope1_tco2e")
    )
)

display(scope1)

### Combine Scope 1 and Scope 2 Emissions

In [ ]:
facility_emissions = (
    scope1
    .join(
        scope2,
        ["facility_id", "period_end"],
        "full_outer"
    )
    .fillna({
        "scope1_tco2e": 0,
        "scope2_tco2e": 0,
        "total_electricity_kwh": 0
    })
    .withColumn(
        "total_tco2e",
        F.col("scope1_tco2e") +
        F.col("scope2_tco2e")
    )
    .withColumn("year", F.year("period_end"))
    .withColumn("month", F.month("period_end"))
)

display(facility_emissions)

### Enrich with Facility Information

In [ ]:
facilities = (
    spark.table("carbonops.silver.facilities_clean")
    .select(
        "facility_id",
        "facility_name",
        "city",
        "state",
        "country",
        "industry"
    )
)

facility_emissions = (
    facility_emissions
    .join(facilities, "facility_id", "left")
)

display(facility_emissions)

### Write Gold Facility Emissions Table

In [ ]:
(
    facility_emissions.write
    .format("delta")
    .mode("overwrite")
    .option("overwriteSchema", "true")
    .saveAsTable("carbonops.gold.facility_emissions")
)

print("Gold facility_emissions table created!")

### Calculate Carbon Intensity

Carbon intensity measures how much carbon is emitted per unit of production.
$$
\text{Carbon Intensity} = \frac{\text{Total Emissions (tCO₂e)}\times1000}{\text{Production Units}}
$$

In [ ]:
# Aggregate production by facility and month
production_monthly = (
    production
    .groupBy("facility_id", "period_end")
    .agg(
        F.sum("production_units").alias("production_units"),
        F.sum("operating_hours").alias("operating_hours")
    )
)

# Join emissions with production
carbon_intensity = (
    facility_emissions
    .join(
        production_monthly,
        ["facility_id", "period_end"],
        "left"
    )
    .withColumn(
        "carbon_intensity_kg_per_unit",
        F.when(
            F.col("production_units") > 0,
            F.col("total_tco2e") * 1000 /
            F.col("production_units")
        )
    )
)

display(carbon_intensity)

### Write Carbon Intensity Gold Table

In [ ]:
(
    carbon_intensity.write
    .format("delta")
    .mode("overwrite")
    .option("overwriteSchema", "true")
    .saveAsTable("carbonops.gold.carbon_intensity")
)

print("Gold carbon_intensity table created!")

### Calculate Carbon Target Performance

CarbonOps also needs to monitor whether its facilities are meeting their annual emissions targets.

We'll calculate:
- Annual emissions target
- Year-to-date emissions
- Remaining emissions budget
- Percentage of annual target utilized

In [ ]:
# Calculate cumulative emissions within each year
window_spec = (
    Window
    .partitionBy("facility_id", "year")
    .orderBy("period_end")
    .rowsBetween(
        Window.unboundedPreceding,
        Window.currentRow
    )
)

target_performance = (
    facility_emissions
    .join(
        carbon_targets.select(
            "facility_id",
            "year",
            "annual_target_tco2e"
        ),
        ["facility_id", "year"],
        "left"
    )
    .withColumn(
        "ytd_emissions_tco2e",
        F.sum("total_tco2e").over(window_spec)
    )
    .withColumn(
        "remaining_budget_tco2e",
        F.col("annual_target_tco2e") -
        F.col("ytd_emissions_tco2e")
    )
    .withColumn(
        "target_utilization_pct",
        F.when(
            F.col("annual_target_tco2e") > 0,
            F.round(
                F.col("ytd_emissions_tco2e") /
                F.col("annual_target_tco2e") * 100,
                2
            )
        )
    )
    .withColumn(
        "target_status",
        F.when(
            F.col("annual_target_tco2e").isNull(),
            "No Target"
        )
        .when(
            F.col("ytd_emissions_tco2e") >
            F.col("annual_target_tco2e"),
            "Exceeded"
        )
        .otherwise("Within Annual Budget")
    )
)

display(target_performance)

### Write Target Performance Gold Table

In [ ]:
(
    target_performance.write
    .format("delta")
    .mode("overwrite")
    .option("overwriteSchema", "true")
    .saveAsTable("carbonops.gold.target_performance")
)

print("Gold target_performance table created!")

### Validate Gold Tables

In [ ]:
gold_tables = [
    "facility_emissions",
    "carbon_intensity",
    "target_performance"
]

for table_name in gold_tables:
    df = spark.table(f"carbonops.gold.{table_name}")
    print(f"{table_name}: {df.count()} records")

display(spark.table("carbonops.gold.facility_emissions"))
display(spark.table("carbonops.gold.carbon_intensity"))
display(spark.table("carbonops.gold.target_performance"))